In [6]:
# ============================================================
# 03_classical_ml_v2 - Cell 1
# features.csv load, train/val/test.csv se split, scaler (sirf train pe fit)
# ============================================================

# os module paths ke liye
import os

# numpy aur pandas data handling ke liye
import numpy as np
import pandas as pd

# StandardScaler features ko same scale (mean 0, std 1) pe laane ke liye
from sklearn.preprocessing import StandardScaler

# Project aur dataset folder ke paths
PROJECT_DIR = r"C:\Projects\Internship_Learning"
DATASET_DIR = os.path.join(PROJECT_DIR, "dataset_final")

# features.csv load karta hai (979 rows, 18 columns)
features_df = pd.read_csv(os.path.join(DATASET_DIR, "features_all.csv"))

# Pehle se bane split CSVs load karta hai (train_test_split use NAHI kar rahe)
train_split = pd.read_csv(os.path.join(DATASET_DIR, "train.csv"))
val_split = pd.read_csv(os.path.join(DATASET_DIR, "val.csv"))
test_split = pd.read_csv(os.path.join(DATASET_DIR, "test.csv"))

# Feature columns ki list (filename, label, score, source chhod ke)
FEATURE_COLS = [c for c in features_df.columns if c not in ["filename", "label", "score", "source"]]

# Kaun se features use ho rahe hain, wo print karta hai
print("Features used:", len(FEATURE_COLS))
print(FEATURE_COLS)

# Ek helper function: split CSV ke filenames ke hisaab se features_df se rows nikalta hai
def get_split(split_df):
    # filename pe merge karta hai taaki order split CSV ke hisaab se rahe
    merged = split_df[["filename"]].merge(features_df, on="filename", how="left")
    # Agar koi filename features me nahi mila to NaN aayega, usko check karta hai
    assert merged[FEATURE_COLS].isnull().sum().sum() == 0, "Missing features mile!"
    return merged

# Teeno splits ke liye data nikalta hai
train_df = get_split(train_split)
val_df = get_split(val_split)
test_df = get_split(test_split)

# X (features) aur y (label) alag karta hai
X_train, y_train = train_df[FEATURE_COLS].values, train_df["label"].values
X_val, y_val = val_df[FEATURE_COLS].values, val_df["label"].values
X_test, y_test = test_df[FEATURE_COLS].values, test_df["label"].values

# Scaler sirf TRAIN pe fit hota hai (leakage se bachne ke liye)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)

# Val aur test pe sirf transform (fit nahi)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)

# Shapes print karta hai (685/147/147 aane chahiye)
print("\nTrain:", X_train_s.shape, "| Val:", X_val_s.shape, "| Test:", X_test_s.shape)

# Har split me class distribution print karta hai
for name, y in [("train", y_train), ("val", y_val), ("test", y_test)]:
    print(name, pd.Series(y).value_counts().to_dict())

# Overlap check: teeno splits me koi common filename nahi hona chahiye
tr, va, te = set(train_df["filename"]), set(val_df["filename"]), set(test_df["filename"])
print("\nOverlap train-val:", len(tr & va), "| train-test:", len(tr & te), "| val-test:", len(va & te))

Features used: 22
['brightness', 'contrast', 'edge_density', 'dominant_color_count', 'whitespace_ratio', 'sharpness', 'saturation_mean', 'dark_area_ratio', 'light_area_ratio', 'clutter_score', 'word_count', 'text_coverage_ratio', 'avg_text_x', 'avg_text_y', 'num_blocks', 'avg_block_size', 'block_density', 'lr_balance', 'tb_balance', 'text_like_regions', 'image_like_regions', 'layout_clutter_score']

Train: (685, 22) | Val: (147, 22) | Test: (147, 22)
train {'average': 275, 'good': 223, 'poor': 187}
val {'average': 59, 'good': 48, 'poor': 40}
test {'average': 59, 'good': 48, 'poor': 40}

Overlap train-val: 0 | train-test: 0 | val-test: 0


In [7]:
# ============================================================
# 03_classical_ml_v2 - Cell 2
# LR / RF / SVM baseline: train pe fit, VAL pe evaluate (test abhi nahi)
# ============================================================

# Models sklearn se import karta hai
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

# Metrics: accuracy, macro F1 aur confusion matrix ke liye
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

# Class order fix karta hai taaki confusion matrix ke rows/columns hamesha good, average, poor ke order me rahein
CLASSES = ["good", "average", "poor"]

# Teeno baseline models ki dictionary
# class_weight="balanced" class imbalance (average zyada, poor kam) ko handle karta hai
# random_state=42 taaki result har baar same aaye
models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=42),
    "SVM (RBF)": SVC(kernel="rbf", C=1.0, gamma="scale", class_weight="balanced", random_state=42),
}

# Val results yahan jama honge
val_results = []

# Har model pe loop chalata hai
for name, model in models.items():

    # Model ko TRAIN data pe fit karta hai (scaled features)
    model.fit(X_train_s, y_train)

    # VAL data pe predict karta hai
    y_pred = model.predict(X_val_s)

    # Accuracy aur macro F1 nikalta hai (macro F1 = main metric)
    acc = accuracy_score(y_val, y_pred) * 100
    f1 = f1_score(y_val, y_pred, average="macro") * 100

    # Result list me add karta hai
    val_results.append({"Model": name, "Val Acc (%)": round(acc, 2), "Val Macro F1 (%)": round(f1, 2)})

    # Confusion matrix print karta hai (rows = true, columns = predicted; order good/average/poor)
    print(name)
    print(confusion_matrix(y_val, y_pred, labels=CLASSES))
    print()

# Sab models ka comparison table banata hai aur print karta hai
val_table = pd.DataFrame(val_results)
print(val_table.to_string(index=False))

Logistic Regression
[[26  9 13]
 [23 21 15]
 [13 10 17]]

Random Forest
[[25 11 12]
 [18 31 10]
 [ 9 14 17]]

SVM (RBF)
[[24 10 14]
 [19 26 14]
 [ 8 13 19]]

              Model  Val Acc (%)  Val Macro F1 (%)
Logistic Regression        43.54             43.23
      Random Forest        49.66             48.98
          SVM (RBF)        46.94             46.77


In [9]:
# ============================================================
# 03_classical_ml_v2 - Fix Cell (Cell 3 se pehle chalana)
# y labels ko Arrow array se plain numpy string array me convert karta hai
# ============================================================

# Labels ko numpy ke normal string array me convert karta hai (sklearn CV iske saath sahi chalta hai)
y_train = np.array(y_train, dtype=str)
y_val = np.array(y_val, dtype=str)
y_test = np.array(y_test, dtype=str)

# Type aur class counts print karke verify karta hai
print("Types:", type(y_train), y_train.dtype)
print("train:", pd.Series(y_train).value_counts().to_dict())
print("val:", pd.Series(y_val).value_counts().to_dict())
print("test:", pd.Series(y_test).value_counts().to_dict())

Types: <class 'numpy.ndarray'> <U7
train: {'average': 275, 'good': 223, 'poor': 187}
val: {'average': 59, 'good': 48, 'poor': 40}
test: {'average': 59, 'good': 48, 'poor': 40}


In [10]:
# ============================================================
# 03_classical_ml_v2 - Cell 3
# Hyperparameter tuning: TRAIN pe 5-fold CV (macro F1), phir VAL pe check
# ============================================================

# GridSearchCV: parameter combinations try karke best chunta hai
# StratifiedKFold: har fold me class ratio same rakhta hai
from sklearn.model_selection import GridSearchCV, StratifiedKFold

# Time measure karne ke liye
import time

# 5-fold stratified CV (shuffle=True, seed 42 taaki result repeat ho sake)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Har model ke liye parameter grid (chhota rakha hai taaki jaldi chale)
search_space = {
    "Logistic Regression": (
        LogisticRegression(max_iter=3000, class_weight="balanced", random_state=42),
        {"C": [0.01, 0.1, 1, 10, 100]},
    ),
    "Random Forest": (
        RandomForestClassifier(class_weight="balanced", random_state=42),
        {"n_estimators": [200, 500], "max_depth": [None, 5, 10], "min_samples_leaf": [1, 3, 5]},
    ),
    "SVM (RBF)": (
        SVC(kernel="rbf", class_weight="balanced", random_state=42),
        {"C": [0.1, 1, 10, 100], "gamma": ["scale", 0.01, 0.1]},
    ),
}

# Best (tuned) models yahan store honge (baad me test evaluation me kaam aayenge)
best_models = {}

# Tuned results ki list
tuned_results = []

# Har model pe loop
for name, (model, grid) in search_space.items():

    # Timer start
    t0 = time.time()

    # GridSearch setup: scoring macro F1, saare CPU cores use (n_jobs=-1)
    gs = GridSearchCV(model, grid, scoring="f1_macro", cv=cv, n_jobs=-1)

    # Sirf TRAIN data pe fit (CV andar hi hota hai)
    gs.fit(X_train_s, y_train)

    # Best model (poore train pe refit hua) store karta hai
    best = gs.best_estimator_
    best_models[name] = best

    # VAL pe predict karke check karta hai
    y_pred = best.predict(X_val_s)
    val_acc = accuracy_score(y_val, y_pred) * 100
    val_f1 = f1_score(y_val, y_pred, average="macro") * 100

    # Result list me add karta hai
    tuned_results.append({
        "Model": name,
        "Best Params": gs.best_params_,
        "CV F1 (%)": round(gs.best_score_ * 100, 2),
        "Val Acc (%)": round(val_acc, 2),
        "Val Macro F1 (%)": round(val_f1, 2),
    })

    # Progress print
    print(f"{name} done | {time.time() - t0:.1f} sec | best params: {gs.best_params_}")
    print(confusion_matrix(y_val, y_pred, labels=CLASSES))
    print()

# Final comparison table
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 80)
print(pd.DataFrame(tuned_results).to_string(index=False))

Logistic Regression done | 16.4 sec | best params: {'C': 0.01}
[[27 10 11]
 [22 24 13]
 [15 10 15]]

Random Forest done | 20.4 sec | best params: {'max_depth': None, 'min_samples_leaf': 1, 'n_estimators': 200}
[[25 11 12]
 [19 30 10]
 [ 8 15 17]]

SVM (RBF) done | 0.6 sec | best params: {'C': 10, 'gamma': 0.01}
[[23 13 12]
 [21 25 13]
 [ 6 15 19]]

              Model                                                     Best Params  CV F1 (%)  Val Acc (%)  Val Macro F1 (%)
Logistic Regression                                                     {'C': 0.01}      46.51        44.90             44.26
      Random Forest {'max_depth': None, 'min_samples_leaf': 1, 'n_estimators': 200}      51.48        48.98             48.40
          SVM (RBF)                                        {'C': 10, 'gamma': 0.01}      51.82        45.58             45.61


In [11]:
# ============================================================
# 03_classical_ml_v2 - Cell 4
# FINAL TEST evaluation (sirf ek baar): tuned models, source-wise, misclassified audit
# ============================================================

# Classification report (precision/recall/F1 per class) ke liye import
from sklearn.metrics import classification_report

# Display settings
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)

# Test results yahan jama honge
test_results = []

# Har model ke test predictions yahan store honge (audit ke liye)
test_preds = {}

# Cell 3 ke tuned models (train pe fit hue the) pe loop
for name, model in best_models.items():

    # Test data pe predict karta hai (X_test_s scaler se transform ho chuka hai)
    y_pred = model.predict(X_test_s)

    # Predictions store karta hai
    test_preds[name] = y_pred

    # Accuracy aur macro F1 nikalta hai
    acc = accuracy_score(y_test, y_pred) * 100
    f1 = f1_score(y_test, y_pred, average="macro") * 100

    # Result list me add karta hai
    test_results.append({"Model": name, "Test Acc (%)": round(acc, 2), "Test Macro F1 (%)": round(f1, 2)})

    # Confusion matrix aur classification report print karta hai
    print("=" * 60)
    print(name)
    print("Confusion matrix (rows=true, cols=pred; order good/average/poor):")
    print(confusion_matrix(y_test, y_pred, labels=CLASSES))
    print(classification_report(y_test, y_pred, labels=CLASSES, digits=3))

# Test results ka summary table
test_table = pd.DataFrame(test_results)
print("=" * 60)
print("FINAL TEST RESULTS (22 features, tuned):")
print(test_table.to_string(index=False))

# Purane results (200-image dataset, sirf reference, direct comparable nahi)
old_table = pd.DataFrame([
    {"Model": "Logistic Regression", "Old Acc (%)": 35.0, "Old Macro F1 (%)": 35.5},
    {"Model": "Random Forest", "Old Acc (%)": 50.0, "Old Macro F1 (%)": 50.36},
    {"Model": "SVM (RBF)", "Old Acc (%)": 50.0, "Old Macro F1 (%)": 52.42},
])

# Old vs New comparison table
comparison = old_table.merge(test_table, on="Model")
print("\nOLD (200 imgs, tiny test) vs NEW (979 imgs, test=147):")
print(comparison.to_string(index=False))

# ---------- Source-wise accuracy aur macro F1 (n bhi dikhata hai) ----------
print("\nSOURCE-WISE (test set):")
src_rows = []
for name, y_pred in test_preds.items():
    for src in ["old", "b1", "b2", "b3"]:
        # Is source ki test rows ka mask
        mask = (test_df["source"] == src).values
        n = int(mask.sum())
        acc_s = accuracy_score(y_test[mask], y_pred[mask]) * 100
        f1_s = f1_score(y_test[mask], y_pred[mask], average="macro") * 100
        src_rows.append({"Model": name, "Source": src, "n": n, "Acc (%)": round(acc_s, 1), "Macro F1 (%)": round(f1_s, 1)})
print(pd.DataFrame(src_rows).to_string(index=False))

# ---------- Misclassified audit: CV F1 se best model chunta hai ----------
best_name = max(tuned_results, key=lambda r: r["CV F1 (%)"])["Model"]
print("\nAudit model (highest CV F1):", best_name)

# Audit model ke predictions
audit_pred = test_preds[best_name]

# Test rows ka audit DataFrame (filename, true label, score, source, pred, sahi/galat)
audit_df = test_df[["filename", "label", "score", "source"]].copy()
audit_df["pred_" + best_name] = audit_pred
audit_df["correct"] = (audit_df["label"].values == audit_pred)

# Sirf galat predictions
wrong_df = audit_df[~audit_df["correct"]]
print("Wrong predictions:", len(wrong_df), "of", len(audit_df))

# Galat predictions ko CSV me save karta hai (images dekhne ke liye next step me use hoga)
WRONG_CSV = os.path.join(DATASET_DIR, "misclassified_classical_test.csv")
wrong_df.to_csv(WRONG_CSV, index=False)
print("Saved:", WRONG_CSV)

# Pehli 15 galat rows print karta hai
print(wrong_df.head(15).to_string(index=False))

Logistic Regression
Confusion matrix (rows=true, cols=pred; order good/average/poor):
[[23 17  8]
 [15 27 17]
 [15  8 17]]
              precision    recall  f1-score   support

        good      0.434     0.479     0.455        48
     average      0.519     0.458     0.486        59
        poor      0.405     0.425     0.415        40

    accuracy                          0.456       147
   macro avg      0.453     0.454     0.452       147
weighted avg      0.460     0.456     0.457       147

Random Forest
Confusion matrix (rows=true, cols=pred; order good/average/poor):
[[21 22  5]
 [17 20 22]
 [ 9 19 12]]
              precision    recall  f1-score   support

        good      0.447     0.438     0.442        48
     average      0.328     0.339     0.333        59
        poor      0.308     0.300     0.304        40

    accuracy                          0.361       147
   macro avg      0.361     0.359     0.360       147
weighted avg      0.361     0.361     0.361       147